# 16 — Sample + export to C

**Before:** notebook **15** (training).

**This notebook:** generate text, export weights, optional C sample.

**Learning objectives**

- Generate text with untrained or trained V2 model.
- Export weights for C with `RUN_EXPORT=True` after training.
- Run C greedy sample when checkpoint exists.
- Explain train-in-PyTorch, sample-in-C workflow.

**Online course:** run cells **top-to-bottom**. Setup cell must print `data OK`.



1. **PyTorch** `generate()` below (like notebook 9).
2. **Export** weights to a binary checkpoint.
3. **C** greedy sample with `train_v2_tiny -sample -ckpt ...`.


In [ ]:
# --- Setup: find repo root (llm-c-from-scratch or Cursor workbook) ---
import sys
from pathlib import Path


def find_llm_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "llmc" / "__init__.py").is_file():
            return base
        nested = base / "llm-c-from-scratch"
        if (nested / "llmc" / "__init__.py").is_file():
            return nested
    return Path.cwd()


ROOT = find_llm_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from llmc.notebook_utils import c_dir, checkpoint_path, data_path, v2_checkpoint_path

DATA = data_path(ROOT)
CHECKPOINT = checkpoint_path(ROOT)
C_DIR = c_dir(ROOT)
V2_CKPT = v2_checkpoint_path(ROOT)
print("ROOT", ROOT.resolve())
print("data", "OK" if DATA.is_file() else "missing")


In [ ]:
import torch
from llmc.data import CharTokenizer, load_text
from llmc.deepseek_v2 import DeepSeekV2, DeepSeekV2Config
from llmc.sample import generate

text = load_text(DATA)
tok = CharTokenizer.from_text(text)
model = DeepSeekV2(DeepSeekV2Config.tiny(tok.vocab_size, 64))
model.eval()

prompt = "ROMEO:"
ctx = tok.encode_tensor(prompt).unsqueeze(0)
out = generate(model, ctx, max_new_tokens=120, temperature=0.9, top_k=40)
print("--- PyTorch sample ---")
print(tok.decode(out.squeeze().tolist()))


### Export for C (match `train_v2_tiny.c` tiny config)


In [ ]:
RUN_EXPORT = False  # set True after training in notebook 15

# Run once after training (or uses a short train inside the script)
import subprocess
if RUN_EXPORT:
    subprocess.run([
    "python3", "scripts/export_v2_tiny.py",
    "--match-train-c", "--train-steps", "30",
    "-o", "checkpoints/v2_tiny.bin",
], check=False)

else:
    print("Export skipped — set RUN_EXPORT=True after training")


### C sample

```bash
cd c && ./bin/train_v2_tiny -sample -ckpt ../checkpoints/v2_tiny.bin
```

Greedy decoding — train longer in notebook 15 for better text.


In [ ]:
RUN_C = False  # set True to compile/run C smokes

import shutil, subprocess
from pathlib import Path
ckpt = ROOT / "checkpoints" / "v2_tiny.bin"
if ckpt.exists() and shutil.which("make"):
    subprocess.run(["make", "-s", "bin/train_v2_tiny"], cwd=str(C_DIR))
    r = subprocess.run(["./bin/train_v2_tiny", "-sample", "-ckpt", f"../{ckpt}"], cwd=str(C_DIR), capture_output=True, text=True)
    print(r.stdout)


## Phase 5 (later)

Full backward through MLA + MoE in C (see `vendor/llm.c/train_gpt2.c` backward passes).

Until then: **train in PyTorch**, **infer/sample in C** via export.
